In [ ]:
pip install


In [ ]:
import os
from kubernetes import client as k8s_client
from kubeflow.training import KubeflowOrgV1ReplicaSpec, KubeflowOrgV1RunPolicy
from kubeflow.training import TrainingClinet
from kubeflow.training import KubeflowOrgV1TFJob, KubeflowOrgV1TFJobSpec, KubeflowOrgV1PyTorchJob, KubeflowOrgV1PyTorchJobSpec, KubeflowOrgV1MPIJob, KubeflowOrgV1MPIJobSpec
from kubernetes.client import V1ObjectMeta, V1PodSpec, V1PodTemplateSpec, V1LocalObjectReference
from kubeflow.training import constants


class TrainingJobSDK:
    def __init__(self, name, container_name, training_image, namespace="hxz-cabt", command=None, args=None, resources=None):
        self.name = name
        self.namespace = namespace
        self.container_name = container_name
        self.training_image = training_image

        self.command = command or []
        self.args = args or []
        self.client = TrainingClient(namespace=self.namespace)

        self.model_dir = "/train/model"
        self.data_dir = "/train/data"
        self.local_output_dir = "/train/logs"
        self.script_dir = "/train/script"

        # 默认资源配置：允许用户覆盖
        self.resources = resources or {
            "limits": {"cpu": "2", "memory": "4Gi"},
            "requests": {"cpu": "1", "memory": "2Gi"}
        }

    def create_volume(self, name, host_path):
        """创建 Kubernetes 的 Volume"""
        return k8s_client.V1Volume(
            name=name,
            host_path=k8s_client.V1HostPathVolumeSource(path=host_path)
        )

    def create_volume_mount(self, name, mount_path):
        """创建 VolumeMount 以挂载到容器中"""
        return k8s_client.V1VolumeMount(name=name, mount_path=mount_path)

    def generate_args(self, additional_args=None, script_subdir="", model_subdir="", data_subdir="", log_subdir=""):
        """生成 args 参数，包含可选的子目录"""
        base_args = [
            f"{self.script_dir}/{script_subdir}",
            "--log_dir", f"{self.local_output_dir}/{log_subdir}",
            "--data_dir", f"{self.data_dir}/{data_subdir}",
            "--model_dir", f"{self.model_dir}/{model_subdir}"
        ]
        if additional_args:
            base_args.extend(additional_args)
        return base_args

    def create_container(self):
        """创建容器配置，包含资源定义"""
        return k8s_client.V1Container(
            name=self.container_name,
            image=self.training_image,
            command=self.command,
            args=self.args,
            volume_mounts=self.create_volume_mounts(),
            resources=k8s_client.V1ResourceRequirements(
                limits=self.resources.get("limits"),
                requests=self.resources.get("requests")
            )
        )

    def create_volume_mounts(self):
        """生成所有需要挂载的 VolumeMount 列表"""
        return [
            self.create_volume_mount("model-volume", self.model_dir),
            self.create_volume_mount("data-volume", self.data_dir),
            self.create_volume_mount("local-output-volume", self.local_output_dir),
            self.create_volume_mount("script-volume", self.script_dir)
        ]

    def create_volumes(self):
        user_dirname = os.environ.get("JUPYTERHUB_SERVER_NAME")
        nas_dir = "/path"

        model_nas_dir = os.path.join(nas_dir, "models", user_dirname)
        data_nas_dir = os.path.join(nas_dir, "data", user_dirname)
        log_nas_dir = os.path.join(nas_dir, "logs", user_dirname)
        script_nas_dir = os.path.join(nas_dir, "scripts", user_dirname)
        """生成所有需要的 Volume 列表"""
        return [
            self.create_volume("model-volume", model_nas_dir),
            self.create_volume("data-volume", data_nas_dir),
            self.create_volume("local-output-volume", log_nas_dir),
            self.create_volume("script-volume", script_nas_dir)
        ]

    def create_pod_spec(self):
        """创建 PodSpec 配置"""
        container = self.create_container()
        init_containers = self.init_containers if include_init_containers else []
        return V1PodSpec(
            init_containers=init_containers,
            containers=[container],
            volumes=self.create_volumes(),
            node_selector={
                "cabt": "true"
            },
            image_pull_secrets=[V1LocalObjectReference(name="atfm-image")
                                ],
        )

    def create_replica_spec(self, replicas, pod_spec):
        """创建 ReplicaSpec 配置"""
        return KubeflowOrgV1ReplicaSpec(
            replicas=replicas,
            restart_policy="Never",
            template=V1PodTemplateSpec(spec=pod_spec)
        )

    def create_training_job(self, job_type, replicas):
        """创建不同类型的训练作业"""

        if job_type == "TFJob":
            return self._create_tf_job(replicas)
        elif job_type == "PyTorchJob":
            return self._create_pytorch_job(replicas)
        elif job_type == "MPIJob":
            return self._create_mpi_job(replicas)
        else:
            raise ValueError(f"Unsupported job type: {job_type}")

    def _create_tf_job(self, replicas, pod_spec):
        """私有方法：创建 TFJob"""
        pod_spec = self.create_pod_spec(include_init_containers=False)
        return KubeflowOrgV1TFJob(
            api_version=constants.API_VERSION,
            kind=constants.TFJOB_KIND,
            metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
            spec=KubeflowOrgV1TFJobSpec(
                run_policy=KubeflowOrgV1RunPolicy(clean_pod_policy="None"),
                tf_replica_specs={
                    "Chief": self.create_replica_spec(replicas.get("Chief", 1), pod_spec),
                    "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec),
                    "PS": self.create_replica_spec(replicas.get("PS", 1), pod_spec)
                }
            )
        )

    def _create_pytorch_job(self, replicas, pod_spec):
        """私有方法：创建 PyTorchJob"""
        master_pod_spec = self.create_pod_spec(include_init_containers=False)
        worker_pod_spec = self.create_pod_spec(include_init_containers=True)
        return KubeflowOrgV1PyTorchJob(
            api_version=constants.API_VERSION,
            kind=constants.PYTORCHJOB_KIND,
            metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
            spec=KubeflowOrgV1PyTorchJobSpec(
                run_policy=KubeflowOrgV1RunPolicy(clean_pod_policy="None"),
                pytorch_replica_specs={
                    "Master": self.create_replica_spec(replicas.get("Master", 1), master_pod_spec),
                    "Worker": self.create_replica_spec(replicas.get("Worker", 2), worker_pod_spec)
                }
            )
        )

    def _create_mpi_job(self, replicas, pod_spec):
        """私有方法：创建 MPIJob"""
        pod_spec = self.create_pod_spec(include_init_containers=False)
        return KubeflowOrgV1MPIJob(
            api_version=constants.API_VERSION,
            kind=constants.MPIJOB_KIND,
            metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
            spec=KubeflowOrgV1MPIJobSpec(
                run_policy=KubeflowOrgV1RunPolicy(clean_pod_policy="None"),
                mpi_replica_specs={
                    "Launcher": self.create_replica_spec(replicas.get("Launcher", 1), pod_spec),
                    "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec)
                }
            )
        )


In [ ]:
# 使用示例


sdk = TrainingJobSDK(
    name="mnist-training",
    namespace="kubeflow",
    container_name="tensorflow",
    training_image="kubeflow/tf-mnist-with-summaries:latest",
    command=["python"]
)

args = sdk.generate_args(
    script_subdir="mnist_with_summaries.py",  # 如果没有子目录，传递空字符串
    model_subdir="tf/",
    data_subdir="mnist.npz",
    log_subdir="tf/",  # 如果没有子目录，传递空字符串
    additional_args=[
        "--learning_rate=0.01",
        "--batch_size=150"
    ]
)

sdk.args = args

# 定义 TFJob
job = sdk.create_training_job(
    job_type="TFJob",
    replicas={
        "Chief": 1,
        "Worker": 2,
        "PS": 1
    },
)


In [ ]:
# Create the SDK instance
#TODO：GPU
#TODO： gloo


sdk = TrainingJobSDK(
    name="pytorch-dist-mnist-mpi",
    namespace="hxz-cabt",
    container_name="pytorch",
    training_image="kubeflow/pytorch-dist-mnist:latest",

    command=["python"],
    # resources={
    #     "limits": {"nvidia.com/gpu": "1"},
    #     "requests": {"cpu": "1", "memory": "2Gi"}
    # }
    init_containers=[
        k8s_client.V1Container(
            name="init-pytorch",
            image="kubeflow/pytorch-dist-mnist:latest",
            command=["mkdir", "-p", "/tmp/data"],
            resources={"limits": {"nvidia.com/gpu": "1"}, "requests": {"cpu": "1", "memory": "2Gi"}}
        )
    ]
)

args = sdk.generate_args(
    script_subdir="mnist.py",  # 如果没有子目录，传递空字符串
    model_subdir="py/",
    data_subdir="mnist.npz",
    log_subdir="py/",  # 如果没有子目录，传递空字符串
    additional_args=[
        "--save-model",
        "--backend","mpi"
    ]
)

sdk.args = args


# Create the PytorchJob
job = sdk.create_training_job(
    job_type="PytorchJob",
    replicas= {
        "Master": 1,
        "Worker": 1
    },
)



In [ ]:

sdk = TrainingJobSDK(
    name="tensorflow-mnist",
    namespace="hxz-cabt",
    container_name="mpi",
    training_image="horovod/horovod:0.28.1",
    command=["mpirun","-np", "2",
        "--allow-run-as-root",
        "-bind-to", "none",
        "-map-by", "slot",
        "-x", "LD_LIBRARY_PATH",
        "-x", "PATH",
        "-mca", "pml", "ob1",
        "-mca", "btl", "^openib",
        "python"],
)

args = sdk.generate_args(
    script_subdir="mnist.py",  # 如果没有子目录，传递空字符串
    model_subdir="mpi/",
    data_subdir="mnist.npz",
    log_subdir="mpi/",  # 如果没有子目录，传递空字符串
)

sdk.args = args

# 创建 MPIJob 作业
job = sdk.create_training_job(
    job_type="MPIJob",
    replicas={
        "Launcher": 1,
        "Worker": 2
    },
)



In [ ]:
# 提交 MPIJob
training_clinet = TrainingClient(job_kind=constants.MPI_JOB_KIND)
training_clinet.create_job(job)


In [ ]:
# 获取 MPIJob 日志
training_clinet.get_job_log(sdk.name)


In [ ]:
# 删除 MPIJob
training_clinet.delete_job(sdk.name)
